## 4) เทรน! (loco-manipulation — 20,000 รอบ)

- `--env.scene.num-envs 1024` — น้อยกว่า walking (มือ+object กิน VRAM มากกว่า)
- `--agent.max-iterations 20000` — งานยาก ต้องเทรนนาน
- `--agent.save-interval 200` + `--log-root` → เซฟลง **Google Drive**

**ดู metrics:** `position_error` (ระยะ cube→เป้า) ควรลดลง, `at_goal` ควรเพิ่ม

> ถ้าหุ่นล้มบ่อย/ไม่เดิน = reward ต้องจูน (ธรรมดาสำหรับ loco-manip รอบแรกๆ)

## 1) ตรวจว่ามี GPU

ถ้าบรรทัดล่างไม่ขึ้นชื่อ GPU (เช่น Tesla T4) ให้กลับไปตั้ง Runtime ก่อน

In [9]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Tesla T4, 15360 MiB


## 2) ติดตั้ง mjlab

clone repo แล้วติดตั้งแบบ editable (`-e`) จะได้แก้โค้ด/เพิ่ม task ได้
(ใช้เวลาสักครู่)

In [10]:
# clone repo
!if [ ! -d 'mjlab-custom' ]; then git clone -q https://github.com/anunpanya9/mjlab-custom.git; fi
%cd /content/mjlab-custom

import sys, os

# ติดตั้ง uv (pip ธรรมดาอ่าน [tool.uv.sources] ของ mjlab ไม่ได้ → ลง deps ไม่ครบ)
!curl -LsSf https://astral.sh/uv/install.sh | sh
UV = '/usr/local/bin/uv'

# สำคัญ: ลงเข้า Python 'ตัวเดียวกับที่ kernel นี้ใช้' (sys.executable)
# ไม่งั้น uv ลงเข้า /usr แล้ว kernel มองไม่เห็น → No module named mjlab
# เปลี่ยนจากการติดตั้งแบบ editable (-e) เป็นแบบมาตรฐาน
!{UV} pip install --python {sys.executable} . --index-strategy unsafe-best-match

# ลบบรรทัดที่เพิ่ม sys.path ด้วยตนเองออก เนื่องจากเป็นการติดตั้งแบบมาตรฐาน ระบบควรจัดการให้แล้ว

import mjlab
print('✓ ติดตั้ง mjlab เข้า', sys.executable, '— import ได้เลย')

/content/mjlab-custom
downloading uv 0.12.12 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Using Python 3.13.15 environment at: /usr
Resolved 123 packages in 225ms
Prepared 1 package in 2.17s
Uninstalled 1 package in 5ms
Installed 1 package in 6ms
 ~ mjlab==1.6.0 (from file:///content/mjlab-custom)
✓ ติดตั้ง mjlab เข้า /usr/bin/python3 — import ได้เลย


## 3) ปิด Weights & Biases (ให้เทรนได้เลยไม่ต้อง login)

mjlab ใช้ W&B log การเทรน. ตั้ง offline เพื่อข้ามการ login (ผลเทรนยังเซฟใน
เครื่องปกติ). ถ้าอยากดู dashboard ออนไลน์ ให้ `!wandb login` แทน

In [11]:
!wandb offline

wandb: Updated settings file /content/mjlab-custom/wandb/settings
W&B offline. Running your script from this directory will only write metadata locally. Use `wandb disabled` to completely turn off W&B.


## 4) เทรน! (loco-manipulation — 20,000 รอบ)

- `--env.scene.num-envs 1024` — น้อยกว่า walking (มือ+object กิน VRAM มากกว่า)
- `--agent.max-iterations 20000` — งานยาก ต้องเทรนนาน
- `--agent.save-interval 200` + `--log-root` → เซฟลง **Google Drive**

**ดู metrics:** `position_error` (ระยะ cube→เป้า) ควรลดลง, `at_goal` ควรเพิ่ม

> ถ้าหุ่นล้มบ่อย/ไม่เดิน = reward ต้องจูน (ธรรมดาสำหรับ loco-manip รอบแรกๆ)

## 4) เทรน! (loco-manipulation — 20,000 รอบ)

- `--env.scene.num-envs 1024` — น้อยกว่า walking (มือ+object กิน VRAM มากกว่า)
- `--agent.max-iterations 20000` — งานยาก ต้องเทรนนาน
- `--agent.save-interval 200` + `--log-root` → เซฟลง **Google Drive**

**ดู metrics:** `position_error` (ระยะ cube→เป้า) ควรลดลง, `at_goal` ควรเพิ่ม

> ถ้าหุ่นล้มบ่อย/ไม่เดิน = reward ต้องจูน (ธรรมดาสำหรับ loco-manip รอบแรกๆ)

In [12]:
from google.colab import drive
drive.mount('/content/drive')

import os
# โฟลเดอร์เก็บผลเทรนบน Drive
LOG_ROOT = '/content/drive/MyDrive/mjlab_logs'
os.makedirs(LOG_ROOT, exist_ok=True)
print('checkpoint จะเซฟที่:', LOG_ROOT)

Mounted at /content/drive
checkpoint จะเซฟที่: /content/drive/MyDrive/mjlab_logs


In [ ]:
import os
os.environ['LOG_ROOT_ENV'] = LOG_ROOT

!python -m mjlab.scripts.train Mjlab-LocoManip-Unitree-G1 \
    --env.scene.num-envs 1024 \
    --agent.max-iterations 20000 \
    --agent.save-interval 200 \
    --agent.logger tensorboard \
    --log-root $LOG_ROOT_ENV

## 4) เทรน! (loco-manipulation — 20,000 รอบ)

- `--env.scene.num-envs 1024` — น้อยกว่า walking (มือ+object กิน VRAM มากกว่า)
- `--agent.max-iterations 20000` — งานยาก ต้องเทรนนาน
- `--agent.save-interval 200` + `--log-root` → เซฟลง **Google Drive**

**ดู metrics:** `position_error` (ระยะ cube→เป้า) ควรลดลง, `at_goal` ควรเพิ่ม

> ถ้าหุ่นล้มบ่อย/ไม่เดิน = reward ต้องจูน (ธรรมดาสำหรับ loco-manip รอบแรกๆ)

In [ ]:
import os
from pathlib import Path

# checkpoint อยู่บน Google Drive (จาก --log-root)
log_dir = Path(LOG_ROOT) / 'g1_locomanip'
runs = sorted(log_dir.glob('*'), key=os.path.getmtime, reverse=True)
assert runs, 'ไม่พบ run บน Drive — เทรนสำเร็จหรือยัง?'
latest = runs[0]
ckpts = sorted(latest.glob('model_*.pt'),
               key=lambda p: int(''.join(filter(str.isdigit, p.stem))))
checkpoint = str(ckpts[-1])
print('run:', latest.name)
print('checkpoints ล่าสุด:', [c.name for c in ckpts[-3:]])
print('✅ โมเดล:', checkpoint)
print('   (เซฟบน Google Drive แล้ว)')

## 6) ทดสอบโมเดล — ให้ policy ที่เทรนแล้วสั่งหุ่น แล้วอัดวิดีโอ

**แนวคิด:** โหลด checkpoint กลับเข้ามาเป็น policy แล้วให้มันสั่งหุ่นจริง (ไม่ใช่
random แล้ว!) เราเรนเดอร์ทีละเฟรมเองเป็นวิดีโอ — วิธีนี้ควบคุมได้เต็มที่และ
**ไม่ค้าง** (ไม่เปิด viewer ที่ Colab ไม่มีจอ)

> ตั้ง `MUJOCO_GL=egl` เพื่อเรนเดอร์แบบ headless (ไม่ต้องมีจอ) — ต้องตั้ง
> **ก่อน** import mujoco/สร้าง env ครั้งแรก

In [ ]:
os.environ["MUJOCO_GL"] = "egl"  # headless render บน Colab

from dataclasses import asdict

import imageio
import torch

import mjlab.tasks  # noqa: F401
from mjlab.envs import ManagerBasedRlEnv
from mjlab.rl import RslRlVecEnvWrapper
from mjlab.rl.runner import MjlabOnPolicyRunner
from mjlab.tasks.registry import load_env_cfg, load_rl_cfg, load_runner_cls

TASK = "Mjlab-LocoManip-Unitree-G1"
device = "cuda" if torch.cuda.is_available() else "cpu"

# สร้าง env แบบ play (1 ตัว) พร้อม render_mode
env_cfg = load_env_cfg(TASK, play=True)
env_cfg.scene.num_envs = 1
eval_env = ManagerBasedRlEnv(cfg=env_cfg, device=device, render_mode="rgb_array")

# โหลด policy จาก checkpoint
agent_cfg = load_rl_cfg(TASK)
runner_cls = load_runner_cls(TASK) or MjlabOnPolicyRunner
wrapped = RslRlVecEnvWrapper(eval_env, clip_actions=agent_cfg.clip_actions)
runner = runner_cls(wrapped, asdict(agent_cfg), device=device)
runner.load(checkpoint, load_cfg={"actor": True}, strict=True, map_location=device)
policy = runner.get_inference_policy(device=device)
print("✓ โหลด policy จาก", Path(checkpoint).name)

In [ ]:
# rollout: ให้ policy สั่งหุ่น 200 step แล้วเก็บเฟรมเป็นวิดีโอ
obs = wrapped.get_observations()
frames = []
for step in range(200):
  with torch.inference_mode():
    action = policy(obs)
  obs, _, _, _ = wrapped.step(action)
  frames.append(eval_env.render())  # rgb array (H, W, 3)

VIDEO_PATH = os.path.join(LOG_ROOT, 'g1_locomanip.mp4')
out = VIDEO_PATH
imageio.mimsave(out, frames, fps=30)
print(f"✓ อัดวิดีโอ {len(frames)} เฟรม -> {out}")

In [ ]:
from IPython.display import Video

Video(VIDEO_PATH, embed=True, width=480)

## 7) ⬇️ ดาวน์โหลดโมเดลไปใช้งาน

ไฟล์ `.pt` นี้คือ **โมเดลที่ใช้งานได้จริง** — เอาไปโหลดที่เครื่องอื่น
(ที่มี mjlab) แล้วสั่งหุ่นด้วย `play --checkpoint-file <ไฟล์>` ได้เลย

In [ ]:
from google.colab import files

print("กำลังดาวน์โหลด:", checkpoint)
files.download(checkpoint)

## 4) เทรน! (loco-manipulation — 20,000 รอบ)

- `--env.scene.num-envs 1024` — น้อยกว่า walking (มือ+object กิน VRAM มากกว่า)
- `--agent.max-iterations 20000` — งานยาก ต้องเทรนนาน
- `--agent.save-interval 200` + `--log-root` → เซฟลง **Google Drive**

**ดู metrics:** `position_error` (ระยะ cube→เป้า) ควรลดลง, `at_goal` ควรเพิ่ม

> ถ้าหุ่นล้มบ่อย/ไม่เดิน = reward ต้องจูน (ธรรมดาสำหรับ loco-manip รอบแรกๆ)